# Notebook 07 — Phase 4: Top-10 Worst Residuals & Molecular Identity Audit

**Purpose.** The Phase 4 README cites a top-10 worst residuals table whose molecular identities are best-guesses from SMILES inspection (e.g. "coronene", "mirex-like organochlorine", "riboflavin"). This notebook does three things:

1. **Reproduces the top-10 pooled across all 5 seeds** by re-fitting the Phase 4 MLP on each seed using the canonical `best_params` from `reports/phase4_summary.json`. The Optuna study is not re-run; only the final 5 fits.
2. **Verifies the molecular identity** of each entry via RDKit canonicalization, InChIKey computation, and SMARTS-based chemical class flagging.
3. **Persists the authoritative top-10 table** to `reports/phase4_worst10.json` and the grid visualization to `reports/figures/04_phase4_worst10.png`. These artifacts become the single source of truth for blog posts, LinkedIn writeups, and the README's worst-10 section.

**Why this matters.** The README explicitly notes the worst-10 names are best-guesses needing verification. Publishing the compression-bias finding externally (LinkedIn / GitHub gist) without verified identities risks a public correction. This notebook closes that gap.

**Runtime.** ~5 minutes for the 5 refits (Colab T4 GPU); ~2 minutes for the RDKit analysis. Total ≤10 minutes.

## 1. Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/qsar-esol-solubility')
else:
    PROJECT_ROOT = Path.cwd().parent

REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors, inchi
RDLogger.DisableLog('rdApp.*')

from src.splits import scaffold_split_balanced
from src.featurization import featurize_morgan
from src.training import train_mlp, seed_everything

RANDOM_SEED = 42
SCAFFOLD_SEEDS = [42, 0, 1, 2, 3]

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")

## 2. Load Phase 4 artifacts + dataset

Read `best_params` from the canonical summary. Verify the dataset is the dedup CSV (1117 mols).

In [ ]:
# Load canonical Phase 4 summary
with open(REPORTS_DIR / 'phase4_summary.json') as fh:
    phase4 = json.load(fh)

best_params = phase4['best_params']
print('Phase 4 best_params:')
for k, v in best_params.items():
    print(f'  {k:<15} = {v}')

# Load deduplicated dataset
df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'esol_dedup.csv')
df['mol'] = df['smiles'].apply(Chem.MolFromSmiles)
assert df['mol'].isna().sum() == 0, 'SMILES parse failures'
print(f'\nLoaded {len(df)} molecules from esol_dedup.csv')
print(f'logS range: [{df["logS"].min():.2f}, {df["logS"].max():.2f}], '
      f'μ = {df["logS"].mean():.2f}, σ = {df["logS"].std():.2f}')

## 3. Re-fit Phase 4 MLP on each seed and collect predictions

For each of the 5 scaffold seeds: rebuild the same train/valid/test split that Phase 4 used, featurize with Morgan FP, train the MLP with `best_params`, and predict on the test set.

This costs ~1 minute/seed on Colab T4 GPU and ~3 min/seed on CPU.

**Sanity check inline:** for seed=42, the test SMILES we generate here must match the `test_smiles` field that Phase 4 stored in the summary JSON. If they don't match, the scaffold split is non-deterministic — which would invalidate the whole comparison.

In [ ]:
# Map hidden_dims from JSON (list) to tuple, since train_mlp's signature wants Sequence
hidden_dims = tuple(best_params['hidden_dims'])

rows = []
for seed in SCAFFOLD_SEEDS:
    print(f'\n--- seed {seed} ---')
    seed_everything(seed)

    tr_idx, va_idx, te_idx = scaffold_split_balanced(df, seed=seed)
    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)
    df_te = df.iloc[te_idx].reset_index(drop=True)

    # Inline sanity check vs Phase 4 stored test SMILES (seed=42 only)
    if seed == 42:
        p4_test_smiles = [r['test_smiles'] for r in phase4['per_seed'] if r['seed'] == 42][0]
        regen_test_smiles = df_te['smiles'].tolist()
        if set(p4_test_smiles) == set(regen_test_smiles):
            print(f'  ✓ test SMILES (seed=42) match Phase 4 stored value ({len(p4_test_smiles)} mols)')
        else:
            n_intersect = len(set(p4_test_smiles) & set(regen_test_smiles))
            print(f'  ⚠ test SMILES mismatch: {n_intersect}/{len(p4_test_smiles)} overlap. '
                  'Scaffold split may be non-deterministic.')

    X_tr = featurize_morgan(df_tr).astype(np.float32)
    X_va = featurize_morgan(df_va).astype(np.float32)
    X_te = featurize_morgan(df_te).astype(np.float32)
    y_tr = df_tr['logS'].values
    y_va = df_va['logS'].values
    y_te = df_te['logS'].values

    result = train_mlp(
        X_train=X_tr, y_train=y_tr,
        X_valid=X_va, y_valid=y_va,
        hidden_dims=hidden_dims,
        dropout=best_params['dropout'],
        use_batchnorm=best_params['use_batchnorm'],
        lr=best_params['lr'],
        weight_decay=best_params['weight_decay'],
        batch_size=best_params['batch_size'],
        max_epochs=200,
        patience=20,
        seed=seed,
        verbose=False,
    )
    y_pred = result['predict'](X_te)

    test_rmse = float(np.sqrt(np.mean((y_te - y_pred) ** 2)))
    print(f'  test RMSE = {test_rmse:.3f}  (stopped at epoch {result["stopped_at"]})')

    for smi, yt, yp in zip(df_te['smiles'], y_te, y_pred):
        rows.append({
            'seed': seed,
            'smiles': smi,
            'y_true': float(yt),
            'y_pred': float(yp),
            'residual': float(yp - yt),
            'abs_residual': float(abs(yp - yt)),
        })

preds_df = pd.DataFrame(rows)
print(f'\nTotal pooled test predictions: {len(preds_df)} (= 5 × 113)')
print(f'Pooled RMSE: {np.sqrt((preds_df["residual"]**2).mean()):.3f}')

## 4. Top-10 worst residuals (pooled across all seeds)

Same protocol as the README: sort by `|residual|` descending across the union of all 5 test folds, take the top 10. Each molecule may appear up to 5 times in the pool (if it lands in the test fold for multiple seeds), but in practice the balanced scaffold split spreads things out so most molecules appear 0–2 times.

In [ ]:
top10 = preds_df.sort_values('abs_residual', ascending=False).head(10).reset_index(drop=True)

# Inspect appearance frequency for context
appearance_counts = preds_df['smiles'].value_counts()
top10['n_appearances_in_pool'] = top10['smiles'].map(appearance_counts)

print('Top-10 worst residuals (pooled 5-seed):')
print(top10[['seed', 'smiles', 'y_true', 'y_pred', 'residual', 'n_appearances_in_pool']]
      .to_string(index=False))

## 5. Molecular identity via RDKit

For each top-10 molecule compute:
- **Canonical SMILES** and **InChIKey** (unique structure identifiers; InChIKey is the standard cross-database key)
- **MolWt, NumRings, NumAromaticRings, NumHeavyAtoms, NumChlorines, NumOH** (structural descriptors that justify the chemical class label)
- **Chemical class flag** via SMARTS pattern matching against a curated list of relevant classes for this dataset (PAH, organochlorine, steroid, sugar/polyol, succinimide, sulfonamide, stilbene/diaryl-ethene, riboflavin-like isoalloxazine, etc.)

The SMARTS flags are conservative: a molecule can match zero or multiple classes. The output gives a verified, RDKit-grounded class label to replace the README's free-text best guesses.

In [ ]:
# Curated SMARTS catalogue — chemical classes most relevant to ESOL extremes.
# Keep patterns intentionally conservative: false positives are worse than
# false negatives here, because we'd rather flag a molecule as 'unclassified'
# than mislabel it in a published table.
SMARTS_CLASSES = {
    'PAH (≥4 fused aromatic rings)': '[c]1[c][c][c]2[c]([c]1)[c][c][c]3[c]2[c][c][c]4[c]3[c][c][c][c]4',
    'PAH (≥3 fused aromatic rings, looser)': 'c1ccc2cc3ccccc3cc2c1',
    'Polychloro (≥4 Cl atoms)': '[Cl].[Cl].[Cl].[Cl]',
    'Steroid backbone (cyclopentanoperhydrophenanthrene)': 'C1CCC2CCC3CCCC4CCCCC1C2C34',
    'Sugar/polyol (≥3 OH on sp3 C)': '[CX4]([OX2H])[CX4]([OX2H])[CX4]([OX2H])',
    'Glycoside (acetal linkage)': '[OX2][CX4H1]([OX2])[#6]',
    'Succinimide / barbiturate-like (cyclic imide)': 'O=C1NC(=O)CC1',
    'Sulfonamide': '[#6]S(=O)(=O)N',
    'Stilbene/diaryl-ethene': 'c1ccc(/C=C/c2ccccc2)cc1',
    'Diaryl ether': 'c1ccc(Oc2ccccc2)cc1',
    'Isoalloxazine (riboflavin core)': 'c1cc2nc3c(=O)[nH]c(=O)nc-3n(c2cc1)',
    'Nucleoside (pyrimidine/purine + sugar)': '[#7]1[#6]=[#7][#6]=[#6][#6]=1[CX4H1]([OX2])[CX4H1]',
    'Hexachloronorbornene/mirex-like': 'C1(Cl)(Cl)C2(Cl)C(Cl)(Cl)C1(Cl)C2(Cl)Cl',
}

def classify(mol):
    """Return list of chemical class labels matching this molecule."""
    hits = []
    for label, sm in SMARTS_CLASSES.items():
        try:
            patt = Chem.MolFromSmarts(sm)
            if patt is None:
                continue
            if mol.HasSubstructMatch(patt):
                hits.append(label)
        except Exception:
            continue
    return hits

def describe(smiles):
    """Full structural profile for one SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'error': 'SMILES failed to parse'}
    return {
        'canonical_smiles': Chem.MolToSmiles(mol),
        'inchikey': inchi.MolToInchiKey(mol),
        'mol_wt': round(Descriptors.MolWt(mol), 1),
        'n_heavy_atoms': mol.GetNumHeavyAtoms(),
        'n_rings': Descriptors.RingCount(mol),
        'n_aromatic_rings': Descriptors.NumAromaticRings(mol),
        'n_Cl': sum(1 for a in mol.GetAtoms() if a.GetSymbol() == 'Cl'),
        'n_OH': sum(
            1 for a in mol.GetAtoms()
            if a.GetSymbol() == 'O' and a.GetTotalNumHs() >= 1
        ),
        'logP_crippen': round(Descriptors.MolLogP(mol), 2),
        'tpsa': round(Descriptors.TPSA(mol), 1),
        'classes': classify(mol),
    }

# Profile each top-10 molecule
profiles = [describe(s) for s in top10['smiles']]
for i, (row, prof) in enumerate(zip(top10.itertuples(index=False), profiles), start=1):
    print(f'\n[{i}] residual = {row.residual:+.2f}  (y_true={row.y_true:+.2f}, y_pred={row.y_pred:+.2f})')
    print(f'    SMILES      : {row.smiles}')
    print(f'    InChIKey    : {prof["inchikey"]}')
    print(f'    Descriptors : MW={prof["mol_wt"]}  heavy={prof["n_heavy_atoms"]}  '
          f'rings={prof["n_rings"]} (arom={prof["n_aromatic_rings"]})  '
          f'Cl={prof["n_Cl"]}  OH={prof["n_OH"]}  logP={prof["logP_crippen"]}  TPSA={prof["tpsa"]}')
    print(f'    Classes     : {prof["classes"] if prof["classes"] else "[unclassified]"}')

## 6. Comparison with the README's hand-curated identities

Side-by-side check. The README lists 10 identities from SMILES inspection; verify how many of the *new* RDKit-derived top-10 overlap (by canonical SMILES / InChIKey) with the README's list. Two failure modes to watch for:

- **Identity drift:** the README labels a molecule "Coronene" but the SMARTS-based classifier disagrees → label was wrong.
- **Set drift:** the top-10 has shifted entirely (e.g. a different seed contributed a worse outlier than expected) → the README's set itself needs updating.

**Set drift is more likely than identity drift** because the README's original top-10 was extracted on a single seed or a slightly different split, and PyTorch GPU non-determinism may shift the 9th–10th ranked positions even if the top-5 are stable.

In [ ]:
# README's free-text identities for the top-10 (in order, as published).
# Re-paste here so the comparison is explicit.
readme_top10_descriptions = [
    ('Coronene (PAH, 7 fused rings)',         -9.33, -4.24, +5.09),
    ('Mirex-like organochlorine',             -6.80, -2.24, +4.56),
    ('Lipophilic diaryl ether',               -8.60, -4.36, +4.24),
    ('Triterpene/steroid',                    -7.32, -3.85, +3.47),
    ('Sulfonamide (piroxicam-like)',          -4.16, -0.74, +3.42),
    ('Succinimide (small, polar)',            +0.30, -3.08, -3.38),
    ('Polycyclic aromatic hydrocarbon',       -8.49, -5.11, +3.38),
    ('Benzopyrene-like PAH',                  -9.02, -5.72, +3.30),
    ('Hexachlorocyclopentadiene-like',        -7.28, -4.03, +3.24),
    ('Diethylstilbestrol/stilbene',           -4.95, -1.80, +3.15),
]

# Match by y_true (chemical identity), tolerating tiny float diffs.
def find_readme_match(y_true_target, tol=0.02):
    for desc, yt, yp, res in readme_top10_descriptions:
        if abs(yt - y_true_target) < tol:
            return desc, yt, yp, res
    return None

print(f'{"#":<2} {"y_true":>7} {"new resid":>10} | README identity'.ljust(60) + '| RDKit classes')
print('-' * 130)
for i, (row, prof) in enumerate(zip(top10.itertuples(index=False), profiles), start=1):
    readme_match = find_readme_match(row.y_true)
    readme_id = readme_match[0] if readme_match else '(not in README top-10)'
    classes_str = ', '.join(prof['classes']) if prof['classes'] else '[unclassified]'
    print(f'{i:<2} {row.y_true:>+7.2f} {row.residual:>+10.2f} | {readme_id:<55} | {classes_str}')

## 7. Grid visualization

2D structural depiction of all 10 molecules with residual annotations. Saved as PNG to `reports/figures/04_phase4_worst10.png` for inclusion in the README or external posts.

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in top10['smiles']]
legends = [
    f'#{i+1}  y_true={r.y_true:+.2f}\ny_pred={r.y_pred:+.2f}  res={r.residual:+.2f}'
    for i, r in enumerate(top10.itertuples(index=False))
]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=5,
    subImgSize=(280, 240),
    legends=legends,
    useSVG=False,
)
fig_path = FIGURES_DIR / '04_phase4_worst10.png'
img.save(fig_path)
print(f'Saved: {fig_path}')
img

## 8. Persist the authoritative top-10 table

Write `reports/phase4_worst10.json` so downstream consumers (the README, future blog posts, the LinkedIn writeup) can cite this analysis without re-running the notebook.

In [ ]:
out = {
    'phase': 'phase4_mlp_morgan',
    'analysis': 'top10_worst_residuals_pooled_5seed',
    'protocol': {
        'scaffold_seeds': SCAFFOLD_SEEDS,
        'pool_size': int(len(preds_df)),
        'best_params_source': 'reports/phase4_summary.json',
        'best_params': best_params,
    },
    'top10': [
        {
            'rank': i + 1,
            'seed_of_appearance': int(row.seed),
            'smiles': row.smiles,
            'canonical_smiles': prof['canonical_smiles'],
            'inchikey': prof['inchikey'],
            'y_true': row.y_true,
            'y_pred': row.y_pred,
            'residual': row.residual,
            'descriptors': {
                'mol_wt': prof['mol_wt'],
                'n_heavy_atoms': prof['n_heavy_atoms'],
                'n_rings': prof['n_rings'],
                'n_aromatic_rings': prof['n_aromatic_rings'],
                'n_Cl': prof['n_Cl'],
                'n_OH': prof['n_OH'],
                'logP_crippen': prof['logP_crippen'],
                'tpsa': prof['tpsa'],
            },
            'rdkit_classes': prof['classes'],
        }
        for i, (row, prof) in enumerate(zip(top10.itertuples(index=False), profiles))
    ],
}

out_path = REPORTS_DIR / 'phase4_worst10.json'
out_path.write_text(json.dumps(out, indent=2))
print(f'Saved: {out_path}')
print(f'  {len(out["top10"])} molecules profiled')

## 9. Markdown block for README replacement

Generate a drop-in replacement table for the README's `### What the top-10 worst residuals look like` section. The new table now carries InChIKey and a verified class column.

**Action item**: open `README.md`, find the existing top-10 table (under heading `What the top-10 worst residuals look like`), and replace it with the printed block below.

In [ ]:
md = []
md.append('| # | Class (RDKit-verified) | InChIKey (first 14) | y_true | y_pred | Residual |')
md.append('|---|------------------------|---------------------|-------:|-------:|---------:|')
for i, (row, prof) in enumerate(zip(top10.itertuples(index=False), profiles), start=1):
    cls = prof['classes'][0] if prof['classes'] else 'unclassified'
    ikey = prof['inchikey'][:14]
    md.append(
        f'| {i} | {cls} | `{ikey}` | '
        f'{row.y_true:+.2f} | {row.y_pred:+.2f} | '
        f'**{row.residual:+.2f}** |'
    )
md_block = '\n'.join(md)
print(md_block)

(REPORTS_DIR / 'phase4_worst10.md').write_text(md_block + '\n')
print(f'\nSaved: {REPORTS_DIR / "phase4_worst10.md"}')